In [1]:
import os
import numpy as np
import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern, hog
from skimage.measure import shannon_entropy
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import csv

In [2]:
# Set the path to your main dataset folder 
dataset_path = r"D:\Capston Project\Dataset_mansi_500"

In [3]:
# Define defect types
defect_types = [
    "Cutting_Marks", "Hot_tears_cracks", "Inclusion", "Porosity", "Scabs",
    "Shrink", "Surface_Roughness", "Veining", "Wrinkles_Folds_Coldshuts"
]

# Common function to load images from a directory
def load_images(directory):
    images = []
    extensions = (".png", ".jpg", ".jpeg")
    for filename in os.listdir(directory):
        if filename.lower().endswith(extensions):
            img = cv2.imread(os.path.join(directory, filename), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    return images

# Load images for each defect type 
dataset = {defect: load_images(os.path.join(dataset_path, defect)) for defect in defect_types}

In [4]:
# Feature extraction functions
def extract_hog_features(image):
    hog_features, _ = hog(image, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    return hog_features

def extract_lbp_features(image, n_points=24, radius=3):
    lbp = local_binary_pattern(image, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    return hist

def extract_glcm_features(image):
    glcm = graycomatrix(image, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4], symmetric=True, normed=True)
    glcm_features = [
        graycoprops(glcm, 'contrast').mean(),
        graycoprops(glcm, 'dissimilarity').mean(),
        graycoprops(glcm, 'homogeneity').mean(),
        graycoprops(glcm, 'energy').mean(),
        graycoprops(glcm, 'correlation').mean()
    ]
    return glcm_features

def extract_haralick_features(image):
    glcm = graycomatrix(image, [1], [0], 256, symmetric=True, normed=True)
    haralick_features = [
        graycoprops(glcm, 'contrast')[0, 0],
        graycoprops(glcm, 'dissimilarity')[0, 0],
        graycoprops(glcm, 'homogeneity')[0, 0],
        graycoprops(glcm, 'energy')[0, 0],
        graycoprops(glcm, 'correlation')[0, 0],
        graycoprops(glcm, 'ASM')[0, 0]
    ]
    return haralick_features

def extract_entropy(image):
    return shannon_entropy(image)

In [6]:
# Common function to extract all features from an image
def extract_all_features(image):
    hog_feat = extract_hog_features(image)
    lbp_feat = extract_lbp_features(image)
    glcm_feat = extract_glcm_features(image)
    haralick_feat = extract_haralick_features(image)
    entropy = extract_entropy(image)
    all_features = np.concatenate([hog_feat, lbp_feat, glcm_feat, haralick_feat, [entropy]])
    return all_features

In [8]:
# Define the path for the CSV file to save features
csv_file_path = "extracted_features.csv"

# Sample image for feature length calculation
sample_image = next(iter(dataset.values()))[0]  # First image from the dataset
sample_features = extract_all_features(sample_image)  # Extract features once to determine length

with open(csv_file_path, mode='w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    # Write the header with feature names
    header = ["Label"] + [f"HOG_{i}" for i in range(len(extract_hog_features(sample_image)))] + \
             [f"LBP_{i}" for i in range(26)] + \
             ["GLCM_contrast", "GLCM_dissimilarity", "GLCM_homogeneity", "GLCM_energy", "GLCM_correlation"] + \
             ["Haralick_contrast", "Haralick_dissimilarity", "Haralick_homogeneity", "Haralick_energy", "Haralick_correlation", "Haralick_ASM"] + \
             ["Entropy"]
    csv_writer.writerow(header)

    for defect, images in dataset.items():
        for image in images:
            features = extract_all_features(image)
            csv_writer.writerow([defect_types.index(defect)] + list(features))

print(f"Feature extraction completed and saved to {csv_file_path}.")

Feature extraction completed and saved to extracted_features.csv.


In [9]:
# Load the data from CSV
data = np.loadtxt(csv_file_path, delimiter=',', skiprows=1)
X = data[:, 1:]  # Features
y = data[:, 0]   # Labels

# Apply Polynomial Regression
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

# Train the polynomial regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error of Polynomial Regression: {mse}")

MemoryError: Unable to allocate 167. TiB for an array with shape (2552, 8977397010) and data type float64

In [10]:
print(f"The CSV file will be saved at: {os.path.abspath(csv_file_path)}")

The CSV file will be saved at: d:\Capston Project\extracted_features.csv
